# 08 — Robustness Report

Reading-side companion to `run_robustness_all.py`. The script performs the
computations (parallel, seeded, checkpointed); this notebook reads the CSVs
and presents them.

Tests covered: **A** (placebo cross-asset), **B** (block bootstrap),
**C** (sensitivity), **D1** (kernel vs bootstrap SE), **D2** (OLS-LP HAC),
**E** (quantile monotonicity), **F** (sub-period exclusions).

This report supersedes the narration/display layer of the original
exploratory notebook `08_robustness.ipynb`, which is not part of the
replication package.

## 0. Setup & execution

In [ ]:
# ── Setup ──
import sys, subprocess, time
from pathlib import Path
sys.path.insert(0, "..")
from config import CFG, ECON_DIR
CFG.ensure_dirs()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style: serif, print-friendly, monochrome (cf. scripts/paper/style.py).
plt.rcParams.update({
    "figure.dpi":      150,
    "savefig.dpi":     300,
    "font.family":     "serif",
    "font.size":       11,
    "axes.titlesize":  12,
    "axes.labelsize":  11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "grid.linewidth":  0.5,
})

# Toggles
FORCE_RERUN   = False    # if True, regenerate all CSVs via the script
SAVE_FIGURES  = False    # if True, also write figures to paper/figures/
N_BOOT        = 1000
N_JOBS        = -1
SEED          = 42

SCRIPT  = Path.cwd().parent / "scripts" / "run_robustness_all.py"
FIG_DIR = CFG.ROOT / "paper" / "figures"
if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

OUTPUTS = {
    "A":  ECON_DIR / "robustness_placebo_fast.csv",
    "B":  ECON_DIR / "robustness_bootstrap_fast.csv",
    "C":  ECON_DIR / "robustness_sensitivity.csv",
    "D1": ECON_DIR / "se_comparison_kernel_bootstrap.csv",
    "D2": ECON_DIR / "ols_lp_hac_benchmark.csv",
    "E":  ECON_DIR / "quantile_monotonicity_test_fast.csv",
    "F":  ECON_DIR / "robustness_subperiods_fast.csv",
}
ALL_TESTS = list(OUTPUTS.keys())

def _save_fig(fig, name):
    """PDF (LaTeX) + PNG (preview). No-op unless SAVE_FIGURES."""
    if not SAVE_FIGURES:
        return
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    print(f"  saved {name}.pdf + .png")

print("Setup OK")
print(f"  FORCE_RERUN  = {FORCE_RERUN}")
print(f"  SAVE_FIGURES = {SAVE_FIGURES}")

In [ ]:
# ── Auto-execute missing tests ──
missing = [t for t, p in OUTPUTS.items() if not p.exists()]
to_run = ALL_TESTS if FORCE_RERUN else missing

if to_run:
    assert SCRIPT.exists(), SCRIPT
    cmd = [
        sys.executable, str(SCRIPT),
        "--tests",  ",".join(to_run),
        "--n_boot", str(N_BOOT),
        "--n_jobs", str(N_JOBS),
        "--seed",   str(SEED),
    ]
    print("Running:", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=str(Path.cwd().parent), check=True)
    print(f"\nScript done — {(time.time()-t0)/60:.2f} min")
else:
    print("All CSVs present, skipping computation.")
    for t, p in OUTPUTS.items():
        print(f"  [{t}] {p.name}")

## 1. Test A — Placebo cross-asset

β(shock) at three quantiles for ETH / BTC / XRP / DOGE, with the
BTC-orthogonalised liquidation shock. **More negative β = stronger
amplification by DeFi liquidations.** ETH should be the most exposed,
non-DeFi placebos (XRP, DOGE) should be near zero.

In [ ]:
df_A = pd.read_csv(OUTPUTS["A"])
asset_order = ["ETH", "BTC", "XRP", "DOGE"]

print("=" * 65)
print("TEST A: beta(shock) at tau=0.01 — DeFi collateral exposure gradient")
print("=" * 65)
pivot_01 = (df_A[df_A["tau"] == 0.01]
            .pivot_table(index="h", columns="asset", values="beta_shock")
            [asset_order])
print(pivot_01.round(4).to_string())

print("\n" + "=" * 65)
print("beta(shock) at tau=0.50 (median — should be ~0 for all)")
print("=" * 65)
pivot_50 = (df_A[df_A["tau"] == 0.50]
            .pivot_table(index="h", columns="asset", values="beta_shock")
            [asset_order])
print(pivot_50.round(4).to_string())

print("\n--- Sensitivity ratios at tau=0.01 ---")
for h_check in [0, 3, 12]:
    if h_check in pivot_01.index:
        eth_b = pivot_01.loc[h_check, "ETH"]
        for asset in ["BTC", "XRP", "DOGE"]:
            other_b = pivot_01.loc[h_check, asset]
            if other_b != 0 and not np.isnan(other_b):
                print(f"  h={h_check:>2}: ETH/{asset} = {eth_b/other_b:.2f}x")

**Conclusion A.** β(ETH) is markedly more negative than β(BTC) and the
non-DeFi placebos at τ=0.01; β at τ=0.50 is ≈ 0 across all assets. The
amplification gradient lines up with DeFi-collateral exposure, not with
general crypto-market beta.

## 2. Test B — Block bootstrap (τ = 0.01)

In [ ]:
df_B = pd.read_csv(OUTPUTS["B"], index_col="h")

print("=" * 55)
print(f"TEST B: beta(shock) at tau=0.01")
print(f"({N_BOOT} replications, block size = {CFG.ECON.block_boot_size}h)")
print("=" * 55)
print(f"{'h':>3}  {'mean':>8}  {'median':>8}  {'CI 2.5%':>8}  {'CI 97.5%':>8}  {'% neg':>6}")
print("-" * 50)
for h, r in df_B.iterrows():
    print(f"{int(h):>3}  {r['mean']:>8.4f}  {r['median']:>8.4f}  "
          f"{r['ci_lo']:>8.4f}  {r['ci_hi']:>8.4f}  {r['pct_negative']:>5.1f}%")

In [ ]:
# ── Consistency check vs legacy CSV ──
ref_path = ECON_DIR / "robustness_bootstrap.csv"
if ref_path.exists():
    df_B_ref = pd.read_csv(ref_path, index_col="h")
    diff = (df_B[["ci_lo", "ci_hi"]] - df_B_ref[["ci_lo", "ci_hi"]]).abs()
    print("max |Δ| on CI bounds (fast vs legacy):")
    print(diff.max().to_string())
    print("\nCIs match to 2 decimals:",
          bool((diff.round(2) == 0).all().all()))
else:
    print(f"No legacy reference at {ref_path} — skipping comparison.")

**Conclusion B.** Bootstrap CIs are strictly negative on every horizon —
the τ=0.01 effect survives serial-correlation-robust inference. The seeded
refactor matches the legacy CIs to two decimals (sub-bp residual difference
from the change in SeedSequence namespace, expected by CLT at N=1000).

## 3. Test C — Sensitivity to specification choices

Grid of OI thresholds {70, 80, 90} and an alternative shock proxy (collateral
USD seized). Uses the **raw** shock (`log_liq.shift(1)`) — see methodological
note in `run_robustness_all.py`.

In [ ]:
df_C = pd.read_csv(OUTPUTS["C"])

print("=" * 62)
print("TEST C: Sensitivity to Specification Choices (tau=0.01, h=0)")
print("=" * 62)
print(f"{'Specification':<20} {'beta':>10} {'p-val':>8} {'beta_int':>10} {'p-val':>8}")
print("-" * 62)
for _, row in df_C.iterrows():
    sig = "***" if row["pval_shock"] < 0.01 else ("**" if row["pval_shock"] < 0.05 else "")
    print(f"{row['test']:<20} {row['beta_shock']:>10.4f} {row['pval_shock']:>7.4f}{sig:>3} "
          f"{row['beta_interact']:>10.4f} {row['pval_interact']:>8.4f}")

all_neg_C = bool((df_C["beta_shock"] < 0).all())
rng_C = float(df_C["beta_shock"].max() - df_C["beta_shock"].min())
print(f"\nAll beta(shock) negative: {'YES' if all_neg_C else 'NO'}")
print(f"Range: {rng_C:.4f}")

**Conclusion C.** Sign and magnitude of β(shock) are stable across OI
thresholds and across the collateral-USD proxy. The headline result is not
an artefact of the discretisation choice.

## 4. Test D1 — Kernel SE vs Bootstrap SE

In [ ]:
df_D1 = pd.read_csv(OUTPUTS["D1"]).sort_values("h").reset_index(drop=True)

print("═══ TEST D1: SE Comparison at τ = 0.01 ═══\n")
print(f"{'h':>3}  {'β(shock)':>10}  {'SE(kern)':>10}  {'SE(boot)':>10}  "
      f"{'ratio':>7}  {'boot 95% CI':>24}  {'0 ∈ CI?':>8}")
print("─" * 88)
for _, r in df_D1.iterrows():
    zero_in = "YES" if r["zero_in_ci"] else "no"
    print(f"{int(r['h']):>3}  {r['beta']:>10.4f}  {r['se_kernel']:>10.4f}  "
          f"{r['se_boot']:>10.4f}  {r['ratio']:>6.2f}x  "
          f"[{r['ci_lo']:>+10.4f}, {r['ci_hi']:>+10.4f}]  {zero_in:>8}")

In [ ]:
# ── Figure V1: SE_kernel vs SE_boot per horizon ──
fig, ax = plt.subplots(figsize=(6, 4))
h = df_D1["h"].astype(int).to_numpy()
x = np.arange(len(h))
w = 0.4

ax.bar(x - w/2, df_D1["se_kernel"], width=w, color="white",
       edgecolor="black", hatch="////", label="SE(kernel)")
ax.bar(x + w/2, df_D1["se_boot"], width=w, color="black",
       edgecolor="black", label="SE(bootstrap)")

ax.set_xticks(x)
ax.set_xticklabels(h)
ax.set_xlabel("Horizon (h)")
ax.set_ylabel("Standard error of β(shock)")
ax.set_title("Kernel vs block-bootstrap SE at τ = 0.01")
ax.legend(frameon=False)
fig.tight_layout()
_save_fig(fig, "figR_D1_se_kernel_vs_bootstrap")
plt.show()

**Conclusion D1.** Ratio SE(boot)/SE(kernel) ≈ 1 at h=0 (no overlap) and
grows with h. Kernel SE understates uncertainty under overlapping
returns; the block bootstrap is the appropriate inference for h > 0.

## 5. Test D2 — OLS-LP + Newey-West HAC

Mean-regression benchmark. OLS β at the conditional **mean** is expected
≈ 0, confirming that the effect is concentrated in the tails.

In [ ]:
df_D2 = pd.read_csv(OUTPUTS["D2"]).sort_values("h").reset_index(drop=True)

print("═══ TEST D2: OLS Local Projections + Newey-West HAC ═══")
print(f"NW lags = max(h+1, {CFG.ECON.nw_lags}).\n")
print(f"{'h':>3}  {'β(shock)':>10}  {'SE(OLS)':>9}  {'SE(HAC)':>9}  "
      f"{'p(HAC)':>9}  {'ratio':>7}  {'sig':>5}")
print("─" * 60)
for _, r in df_D2.iterrows():
    p = r["pval_hac"]
    sig = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else ""))
    print(f"{int(r['h']):>3}  {r['beta_shock_ols']:>10.4f}  {r['se_ols']:>9.4f}  "
          f"{r['se_hac']:>9.4f}  {p:>8.4f}  {r['ratio_hac_ols']:>6.2f}x  {sig:>5}")

In [ ]:
# ── OLS mean β vs QLP β at extreme/median quantiles ──
qlp_path = ECON_DIR / "quantile_lp_results.csv"
if qlp_path.exists():
    df_qlp = pd.read_csv(qlp_path)
    print(f"{'h':>3}  {'OLS mean':>10}  {'QLP τ=0.01':>12}  "
          f"{'QLP τ=0.50':>12}  {'QLP τ=0.95':>12}")
    print("─" * 55)
    for _, r in df_D2.iterrows():
        h = int(r["h"])
        def _q(tau):
            s = df_qlp.loc[(df_qlp["tau"] == tau) & (df_qlp["h"] == h), "beta_shock"]
            return s.values[0] if len(s) else np.nan
        print(f"{h:>3}  {r['beta_shock_ols']:>10.4f}  "
              f"{_q(0.01):>12.4f}  {_q(0.50):>12.4f}  {_q(0.95):>12.4f}")
else:
    print(f"[skipped] {qlp_path.name} not found — run NB07 first to enable"
          " the OLS-vs-QLP comparison table.")

**Conclusion D2.** β(OLS mean) ≈ 0 across horizons; HAC/OLS SE ratios
rise with h (overlap-induced autocorrelation). The asymmetry
β(0.01) ≪ β(0.50) ≈ 0 ≪ β(0.95) confirms a tail-concentrated effect.

## 6. Test E — Quantile monotonicity

Δ_h = β(τ=0.01, h) − β(τ=0.50, h), with β(0.01) and β(0.50) fitted on
the **same** block resample per replication (paired estimate). p-value
is the recentred bootstrap test of H₀: Δ = 0
(see `run_robustness_all.summarize_pair`).

In [ ]:
df_E = pd.read_csv(OUTPUTS["E"]).sort_values("h").reset_index(drop=True)

print("=" * 65)
print("TEST E: Quantile Monotonicity — D = beta(0.01,h) - beta(0.50,h)")
print(f"({N_BOOT} replications, block size = {CFG.ECON.block_boot_size}h)")
print("=" * 65)
print(f"{'h':>3}  {'D_point':>10}  {'D_mean':>10}  {'CI_lo':>10}  "
      f"{'CI_hi':>10}  {'p-val':>8}  {'0inCI?':>6}")
print("-" * 65)
for _, r in df_E.iterrows():
    zero_in = "YES" if r["delta_ci_lo"] <= 0 <= r["delta_ci_hi"] else "no"
    print(f"{int(r['h']):>3}  {r['delta_point']:>10.4f}  {r['delta_mean']:>10.4f}  "
          f"{r['delta_ci_lo']:>10.4f}  {r['delta_ci_hi']:>10.4f}  "
          f"{r['delta_pval']:>8.4f}  {zero_in:>6}")

print("\n--- Marginal stats per τ ---")
print(f"{'h':>3}  {'β01_mean':>10}  {'β01 95% CI':>22}  "
      f"{'β50_mean':>10}  {'β50 95% CI':>22}")
for _, r in df_E.iterrows():
    print(f"{int(r['h']):>3}  {r['beta01_mean']:>10.4f}  "
          f"[{r['beta01_ci_lo']:>+8.4f}, {r['beta01_ci_hi']:>+8.4f}]  "
          f"{r['beta50_mean']:>10.4f}  "
          f"[{r['beta50_ci_lo']:>+8.4f}, {r['beta50_ci_hi']:>+8.4f}]")

In [ ]:
# ── Best-effort comparison vs legacy E CSV ──
# Legacy schema (NB08 cell 19): h, delta_point, ci_lo, ci_hi, pval_two_sided.
# Fast schema renames CI/pval to delta_*; β01/β50 columns are new.
ref_E = ECON_DIR / "quantile_monotonicity_test.csv"
if ref_E.exists():
    ref = pd.read_csv(ref_E).sort_values("h").reset_index(drop=True)
    pairs = {
        "delta_point": ("delta_point", "delta_point"),
        "delta_ci_lo": ("delta_ci_lo", "ci_lo"),
        "delta_ci_hi": ("delta_ci_hi", "ci_hi"),
        "delta_pval":  ("delta_pval",  "pval_two_sided"),
    }
    print("max |Δ| on common columns (fast vs legacy):")
    for label, (a, b) in pairs.items():
        if a in df_E.columns and b in ref.columns:
            d = df_E[a].to_numpy() - ref[b].to_numpy()
            print(f"  {label:<14}  max |Δ| = {np.nanmax(np.abs(d)):.2e}")
    print("  beta01_*, beta50_* : no legacy reference — new in fast version")
else:
    print(f"No legacy reference at {ref_E} — skipping comparison.")

In [ ]:
# ── Figure V2: quantile asymmetry + paired Δ ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})
h = df_E["h"].astype(int).to_numpy()

# Top: β(τ=0.01) and β(τ=0.50) with bootstrap bands.
ax1.fill_between(h, df_E["beta01_ci_lo"], df_E["beta01_ci_hi"],
                 color="black", alpha=0.15, linewidth=0)
ax1.plot(h, df_E["beta01_mean"], color="black", linewidth=1.5,
         marker="o", markersize=4, label=r"$\beta(\tau=0.01)$")
ax1.fill_between(h, df_E["beta50_ci_lo"], df_E["beta50_ci_hi"],
                 color="gray", alpha=0.20, linewidth=0)
ax1.plot(h, df_E["beta50_mean"], color="gray", linewidth=1.5,
         linestyle="--", marker="s", markersize=4, label=r"$\beta(\tau=0.50)$")
ax1.axhline(0, color="black", linewidth=0.5)
ax1.set_ylabel(r"$\beta(\mathrm{shock})$")
ax1.set_title("Quantile asymmetry of the liquidation shock")
ax1.legend(frameon=False, loc="lower right")

# Bottom: paired Δ_h with bootstrap CI bars.
delta = df_E["delta_point"].to_numpy()
lo = df_E["delta_ci_lo"].to_numpy()
hi = df_E["delta_ci_hi"].to_numpy()
yerr = np.vstack([delta - lo, hi - delta])
ax2.errorbar(h, delta, yerr=yerr, fmt="o", color="black",
             ecolor="black", elinewidth=1, capsize=3, markersize=4)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_xlabel("Horizon (h)")
ax2.set_ylabel(r"$\Delta_h$")
ax2.set_xticks(h)

fig.tight_layout()
_save_fig(fig, "figR_E_quantile_asymmetry")
plt.show()

**Conclusion E.** β(τ=0.01) sits well below β(τ=0.50) ≈ 0 across
horizons; the paired Δ is significantly negative at the short horizons
where the liquidation cascade is most active. The left tail responds
strictly more to the shock than the median — a quantile-specific effect,
not a level shift.

## 7. Test F — Sub-period exclusions

Re-estimate dropping Terra (Apr–Jun 2022), FTX (Oct–Dec 2022), USDC
(Feb–Apr 2023) windows. Verifies the result is not driven by a single
stress episode.

In [ ]:
df_F = pd.read_csv(OUTPUTS["F"])
df_F = df_F.sort_values(["period_dropped", "tau", "h"]).reset_index(drop=True)

print("── Head ──")
print(df_F.head(10).to_string(index=False))
print("\n── Tail ──")
print(df_F.tail(10).to_string(index=False))

check = df_F[(df_F["tau"] == 0.01) & (df_F["h"] == 0)][["period_dropped", "beta"]]
print("\n── Condition check: β(0.01, h=0) ──")
print(check.to_string(index=False))
all_neg_F = bool((check["beta"] < 0).all())
print(f"\nAll negative: {'YES — condition satisfied' if all_neg_F else 'NO — RESULT REQUIRES REVIEW'}")

**Conclusion F.** β(0.01, h=0) remains negative under every exclusion —
the effect is not carried by Terra, FTX or USDC alone.

## 8. Summary

In [ ]:
# Build a one-line verdict per test from the loaded artefacts.
summary = []

# A — gradient ETH/BTC at h=0, τ=0.01
if 0 in pivot_01.index:
    eth0 = pivot_01.loc[0, "ETH"]; btc0 = pivot_01.loc[0, "BTC"]
    summary.append(("A",
        f"ETH/BTC ratio at τ=0.01,h=0 = {eth0/btc0:.2f}x; placebos near zero"))

# B — strictly negative CIs?
all_neg_B = bool((df_B["ci_hi"] < 0).all())
summary.append(("B",
    f"bootstrap CIs strictly negative on every horizon: {all_neg_B}"))

# C — sign stability across specs
summary.append(("C",
    f"β(shock)<0 across all OI thresholds and proxies: {all_neg_C} (range {rng_C:.4f})"))

# D1 — max ratio
max_ratio = float(df_D1["ratio"].max())
summary.append(("D1",
    f"max SE(boot)/SE(kernel) = {max_ratio:.2f}x — kernel underestimates at h>0"))

# D2 — OLS mean β closeness to zero
max_abs_ols = float(df_D2["beta_shock_ols"].abs().max())
summary.append(("D2",
    f"max |β(OLS mean)| = {max_abs_ols:.4f}; effect is in the tails, not the mean"))

# E — Δ_h significant at any h
n_sig_E = int((df_E["delta_pval"] < 0.05).sum())
summary.append(("E",
    f"Δ_h<0 with p<0.05 on {n_sig_E}/{len(df_E)} horizons; tail strictly amplifies median"))

# F — condition check
summary.append(("F",
    f"β(0.01,h=0)<0 under every sub-period exclusion: {all_neg_F}"))

print("=" * 78)
print("ROBUSTNESS REPORT — one-line verdicts")
print("=" * 78)
for t, msg in summary:
    print(f"  TEST {t:<2}: {msg}")